# 06 - Feature Engineering

## Customer Intelligence Platform

This notebook creates engineered features that separate this project from
average analyses. Each feature group has a clear business rationale.

### Feature Groups
1. Tenure Features
2. Service Count
3. Revenue Features
4. Customer Profile Features
5. Risk Indicators
6. Interaction Features

---


In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.load_data import load_clean
from src.feature_engineering import (
    create_tenure_features,
    create_service_count,
    create_revenue_features,
    create_customer_profile_features,
    create_risk_indicators,
    create_interaction_features,
    engineer_features,
)
from src.config import TARGET, FEATURES_FILE
from src.utils import df_summary, save_dataframe

%matplotlib inline


In [ ]:
df = load_clean()
df_summary(df, "Clean Data (Input)")


## 1. Tenure Features

**Business Rationale**: Raw tenure in months hides important behavioral patterns.
Bucketing reveals distinct customer lifecycle stages.


In [ ]:
df = create_tenure_features(df)
print("Tenure Group Distribution:")
print(df["tenure_group"].value_counts().sort_index())
print(f"\nNew Customers (<12 months): {df['is_new_customer'].sum():,} ({df['is_new_customer'].mean():.1%})")
print(f"\nChurn by Tenure Group:")
print(df.groupby("tenure_group", observed=True)[TARGET].mean().to_string())


## 2. Service Count

**Business Rationale**: Customers with more services are stickier.
The total count captures engagement depth.


In [ ]:
df = create_service_count(df)
print("Service Count Distribution:")
print(df["service_count"].value_counts().sort_index())
print(f"\nChurn by Service Count:")
print(df.groupby("service_count")[TARGET].mean().to_string())


## 3. Revenue Features

**Business Rationale**: Revenue per month and revenue intensity
capture spending patterns beyond raw charge amounts.


In [ ]:
df = create_revenue_features(df)
print(f"Revenue per Month: mean={df['revenue_per_month'].mean():.2f}, std={df['revenue_per_month'].std():.2f}")
print(f"Revenue Intensity: mean={df['revenue_intensity'].mean():.2f}, std={df['revenue_intensity'].std():.2f}")


## 4. Customer Profile Features

**Business Rationale**: Family status, tech adoption, and service bundle
provide higher-level customer characterization.


In [ ]:
df = create_customer_profile_features(df)
print(f"Has Family: {df['has_family'].sum():,} ({df['has_family'].mean():.1%})")
print(f"\nTech Adoption Score Distribution:")
print(df["tech_adoption_score"].value_counts().sort_index())
print(f"\nService Bundle Distribution:")
print(df["service_bundle"].value_counts())


## 5. Risk Indicators

**Business Rationale**: Combining known risk factors into a composite score
creates a simple risk stratification.


In [ ]:
df = create_risk_indicators(df)
print("Risk Score Distribution:")
print(df["risk_score"].value_counts().sort_index())
print(f"\nChurn by Risk Score:")
print(df.groupby("risk_score")[TARGET].mean().to_string())


The risk score shows a clear monotonic relationship with churn:
higher risk scores mean higher churn rates. This validates our feature design.

## 6. Interaction Features

**Business Rationale**: Capture non-linear relationships between features.


In [ ]:
df = create_interaction_features(df)
print(f"Security Gap (Fiber without security): {df['security_gap'].sum():,} customers")
print(f"  Churn rate: {df[df['security_gap']==1][TARGET].mean():.1%}")
print(f"\nHigh Charge New Customer: {df['high_charge_new_customer'].sum():,} customers")
print(f"  Churn rate: {df[df['high_charge_new_customer']==1][TARGET].mean():.1%}")


## 7. Feature Summary


In [ ]:
df_summary(df, "Feature-Engineered Data")
print(f"\nNew columns added: {len(df.columns) - 20}")
print(f"\nAll columns ({len(df.columns)}):")
for i, col in enumerate(df.columns):
    print(f"  {i+1}. {col}")


## 8. Validate No Leakage


In [ ]:
# Check engineered features for target leakage
from src.config import LEAKAGE_COLUMNS
leaked = [c for c in LEAKAGE_COLUMNS if c in df.columns]
print(f"Leakage columns present: {leaked or 'None - All clear!'}")
print(f"\nTarget column present: {TARGET in df.columns}")


## 9. Save Engineered Features


In [ ]:
save_dataframe(df, FEATURES_FILE)
print(f"\nSaved to: {FEATURES_FILE}")


## Summary

| Feature Group | Features Created | Business Value |
|---------------|-----------------|----------------|
| Tenure | tenure_group, is_new_customer | Customer lifecycle stage |
| Service Count | service_count | Engagement depth |
| Revenue | revenue_per_month, revenue_intensity | Spending patterns |
| Customer Profile | has_family, tech_adoption_score, service_bundle | Customer characterization |
| Risk Indicators | is_m2m, is_fiber, is_e_check, risk_score | Risk stratification |
| Interactions | contract_tenure, security_gap, high_charge_new | Non-linear patterns |

**Total: 15 new features** added to the 20 clean features.
